# 面试问题：怎样观测 Agent Loop，定位慢、贵、循环、工具错误和质量回归？

**一句话回答**：每个 run 建 root span，模型调用、工具、检索、handoff 和 evaluator 是 child span；记录低基数版本/状态/finish reason、token、成本、排队与错误，敏感 Prompt/结果默认不采集或脱敏。用 span tree 计算关键路径、重试/循环、每任务成本和失败归因；保存受控 tool result 可做确定性 trace replay。

本 Notebook 手写 span 模型、树校验、成本和 critical path、循环检测、隐私处理、采样、重放与 SLO 门禁。

In [ ]:
from dataclasses import dataclass,field
from collections import Counter,defaultdict
import hashlib,json,math,re
import numpy as np

SEED126=12601; rng126=np.random.default_rng(SEED126)
assert SEED126==12601
assert hashlib.sha256(b"trace-a").hexdigest()!=hashlib.sha256(b"trace-b").hexdigest()
assert Counter(["tool","tool","llm"])["tool"]==2

## 1. Span 最小合同与 parent-child 因果关系

span 包含 trace/span/parent ID、operation、start/end、status、低基数属性和 events。ID 唯一，子 span 时间应落在父范围；异步 fan-out 共享父但并行。Prompt 正文不是必填观测字段，版本和 hash 往往足以回归。

In [ ]:
@dataclass(frozen=True)
class Span126:
    trace_id:str; span_id:str; parent_id:str|None; operation:str; start:float; end:float; status:str; attrs:dict=field(default_factory=dict)
    def __post_init__(self):
        if not self.trace_id or not self.span_id or self.end<self.start or self.status not in {"ok","error","cancelled"}: raise ValueError("span_contract")
spans126=[Span126("tr1","root",None,"invoke_agent",0,8,"ok",{"agent.version":"v3"}),Span126("tr1","llm1","root","chat",0,2,"ok",{"input_tokens":100,"output_tokens":40,"model":"m1"}),Span126("tr1","tool1","root","execute_tool",2,6,"ok",{"tool":"search","attempt":1}),Span126("tr1","llm2","root","chat",6,8,"ok",{"input_tokens":180,"output_tokens":60,"model":"m1"})]
assert len({s.span_id for s in spans126})==4
assert spans126[0].parent_id is None and spans126[-1].end==8
try: Span126("t","s",None,"x",2,1,"ok"); raise AssertionError("negative duration accepted")
except ValueError as e: assert str(e)=="span_contract"

## 2. 先验证 trace tree 完整性

orphan、重复 ID、跨 trace parent 和负时长会让归因错误。流式/消息系统可能用 links 而非严格 parent，但语义需明确。下面验证简单树和父时间包络；生产允许 clock skew 时使用单调时钟或容差。

In [ ]:
def validate_trace126(spans):
    by={s.span_id:s for s in spans}
    if len(by)!=len(spans): return False,"duplicate"
    roots=[s for s in spans if s.parent_id is None]
    if len(roots)!=1: return False,"root"
    for s in spans:
        if s.parent_id is not None:
            p=by.get(s.parent_id)
            if not p or p.trace_id!=s.trace_id: return False,"orphan"
            if s.start<p.start or s.end>p.end: return False,"time_envelope"
    return True,"valid"
assert validate_trace126(spans126)==(True,"valid")
assert validate_trace126(spans126+[Span126("tr1","x","missing","tool",1,2,"ok")])==(False,"orphan")
assert sum(s.parent_id is None for s in spans126)==1

## 3. Token 与成本按实际响应模型聚合

每次模型调用记录 input/output/cache/reasoning token、模型版本和计价快照；工具记录调用费与外部流量。Agent 成本按任务成功分母报告，不能只看单次模型价格。下面用固定价表计算 run 成本。

In [ ]:
PRICE126={"m1":{"input":2e-6,"output":6e-6}}
def cost126(spans):
    total=0.; breakdown=[]
    for s in spans:
        if s.operation=="chat":
            p=PRICE126[s.attrs["model"]]; c=s.attrs["input_tokens"]*p["input"]+s.attrs["output_tokens"]*p["output"]; total+=c; breakdown.append((s.span_id,c))
    return total,breakdown
total126,breakdown126=cost126(spans126)
assert math.isclose(total126,100*2e-6+40*6e-6+180*2e-6+60*6e-6)
assert len(breakdown126)==2
assert all(c>0 for _,c in breakdown126)

## 4. 墙钟瓶颈看 critical path，不看 duration 总和

并行 child 的 duration 相加会夸大墙钟。对 DAG span links 求最长路径；简单树中可用父子阶段和区间分析。示例再加入两个并行工具，墙钟为较慢工具 4 秒，而工具计算总量是 7 秒。

In [ ]:
parallel_spans126=[Span126("tr2","root2",None,"invoke_agent",0,6,"ok"),Span126("tr2","plan","root2","chat",0,1,"ok",{"input_tokens":10,"output_tokens":10,"model":"m1"}),Span126("tr2","ta","root2","execute_tool",1,5,"ok",{"tool":"a"}),Span126("tr2","tb","root2","execute_tool",1,4,"ok",{"tool":"b"}),Span126("tr2","join","root2","chat",5,6,"ok",{"input_tokens":10,"output_tokens":10,"model":"m1"})]
tool_sum126=sum(s.end-s.start for s in parallel_spans126 if s.operation=="execute_tool"); tool_wall126=max(s.end for s in parallel_spans126 if s.operation=="execute_tool")-min(s.start for s in parallel_spans126 if s.operation=="execute_tool")
assert tool_sum126==7 and tool_wall126==4
assert tool_wall126<tool_sum126
assert parallel_spans126[0].end-parallel_spans126[0].start==6

## 5. 从规范化 action 序列发现 retry 与循环

同一 tool+参数不同 attempt 是 retry；重复 action pattern 或 A↔B 交替提示 loop。先对敏感参数做稳定 hash，再计数，既可检测又不暴露正文。触发阈值后关联 stop reason 与最后错误。

In [ ]:
actions126=[("search",{"q":"x"}),("calc",{"x":1}),("search",{"q":"x"}),("calc",{"x":1}),("search",{"q":"x"})]
def action_key126(a): return a[0]+":"+hashlib.sha256(json.dumps(a[1],sort_keys=True).encode()).hexdigest()[:12]
keys126=[action_key126(a) for a in actions126]
def repeated_cycle126(keys,period=2,repeats=2): return len(keys)>=period*repeats and keys[-period:]*repeats==keys[-period*repeats:]
assert keys126[0]==keys126[2]==keys126[4]
assert repeated_cycle126(keys126[:4],2,2)
assert not repeated_cycle126(keys126[:3],2,2)

## 6. 可观测性不能变成第二份敏感数据库

属性默认记录长度、hash、分类和版本；正文 opt-in、采样、加密并设短 TTL。邮箱、token、账号等先脱敏；高基数 user/query 不做 metric label，避免成本和泄露。访问 trace 本身需要租户 ACL 与审计。

In [ ]:
EMAIL126=re.compile(r"[\w.+-]+@[\w.-]+")
SECRET126=re.compile(r"(?i)(token|password)=\S+")
def redact126(text): return SECRET126.sub(r"\1=[REDACTED]",EMAIL126.sub("[EMAIL]",text))
redacted126=redact126("联系 a@b.com token=abc123")
assert redacted126=="联系 [EMAIL] token=[REDACTED]"
assert "abc123" not in redacted126 and "a@b.com" not in redacted126
assert redact126("普通文本")=="普通文本"

## 7. Trace replay 注入记录的模型/工具结果

回放用于比较新 planner/policy，而不是重做副作用。fixture 按 `(span operation, request hash)` 返回历史结果；未记录请求失败，防止测试悄悄访问网络。可做 counterfactual：固定 tool observations，只替换规划或提示版本。

In [ ]:
fixtures126={("search",hashlib.sha256(b'{"q":"x"}').hexdigest()):{"docs":[1,2]},("chat",hashlib.sha256(b"prompt-v1").hexdigest()):{"action":"finish"}}
def replay_call126(operation,payload_bytes):
    key=(operation,hashlib.sha256(payload_bytes).hexdigest())
    if key not in fixtures126: raise ValueError("unrecorded_call")
    return json.loads(json.dumps(fixtures126[key]))
assert replay_call126("search",b'{"q":"x"}')=={"docs":[1,2]}
assert replay_call126("chat",b"prompt-v1")["action"]=="finish"
try: replay_call126("search",b"new"); raise AssertionError("network fallback")
except ValueError as e: assert str(e)=="unrecorded_call"

## 8. SLO 与质量事件在同一 trace 关联

按任务/slice 报 success、p50/p95、TTFT、token、成本、工具错误、循环和人工升级；高延迟 trace 自动保留结构但仍脱敏。发布比较 paired trace，定位是模型、队列、工具、重试还是 evaluator 导致回归。

In [ ]:
latency126=np.array([1.1,1.3,1.2,4.8,1.0,1.4,1.2,1.1]); p95_126=float(np.quantile(latency126,.95)); error_rate126=1/20
manifest126={"schema":1,"convention":"otel-genai-aligned-demo","root":"invoke_agent","children":["chat","execute_tool","retrieve","handoff","evaluate"],"content":"opt_in_redacted","replay":"recorded_results_only","slo":{"p95":5.,"error_rate":.1}}; digest126=hashlib.sha256(json.dumps(manifest126,sort_keys=True).encode()).hexdigest()
assert p95_126<manifest126["slo"]["p95"] and error_rate126<manifest126["slo"]["error_rate"]
assert manifest126["content"]=="opt_in_redacted"
assert len(digest126)==64

## 面试总结

回答顺序是：**root/child span 合同 → tree 完整性 → token/成本 → parallel critical path → retry/loop fingerprint → PII/高基数治理 → recorded-result replay → 按 slice 的质量/延迟/成本门禁**。Agent 可观测性的目标是从最终回答反推每个决策与外部作用，而不是无边界记录全部 Prompt。

延伸阅读：[OpenTelemetry GenAI Attributes](https://opentelemetry.io/docs/specs/semconv/registry/attributes/gen-ai/)、[OpenTelemetry Trace](https://opentelemetry.io/docs/specs/otel/trace/)、[Agent Evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)。